# **Federated Next-Word Prediction for Typing Suggestions**
**Objective**

Build a Federated Learning NLP system that predicts the next word in a sentence without sharing user text data.

Example:
Input: I like

Prediction: I like → machine / coding / learning

**Step 1 – Install Libraries**

In [ ]:
!pip install torch numpy

**Step 2 – Import Libraries**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import Counter

**Step 3 – Sample Text Dataset**

This simulates text typed by users on different devices.

In [ ]:
sentences = [
"I love machine learning",
"machine learning is powerful",
"I enjoy learning new algorithms",
"deep learning improves prediction",
"federated learning protects privacy",
"privacy preserving machine learning",
"artificial intelligence is the future",
"I love artificial intelligence",
"machine learning models learn patterns"
]

**Step 4 – Text Tokenization**

In [ ]:
def tokenize(sentence):
    return sentence.lower().split()

tokens = []

for s in sentences:
    tokens.extend(tokenize(s))

**Step 5 – Vocabulary Creation**

In [ ]:
vocab = Counter(tokens)

word2idx = {word:i+1 for i,(word,_) in enumerate(vocab.items())}
idx2word = {i:word for word,i in word2idx.items()}

vocab_size = len(word2idx)

**Step 6 – Create Training Pairs**

Example:

I like machine learning

**Training pairs:**

I → like
love → machine
machine → learning

Implementation:

In [ ]:
def create_pairs(sentences):

    pairs = []

    for s in sentences:

        words = tokenize(s)

        for i in range(len(words)-1):

            input_word = word2idx[words[i]]
            target_word = word2idx[words[i+1]]

            pairs.append((input_word,target_word))

    return pairs

pairs = create_pairs(sentences)

**Step 7 – Simulate Federated Clients**

In [ ]:
NUM_CLIENTS = 3

random.shuffle(pairs)

size = len(pairs)//NUM_CLIENTS

client_data = []

for i in range(NUM_CLIENTS):

    start = i*size
    end = start+size

    client_data.append(pairs[start:end])

Each client represents a mobile device keyboard dataset.

**Step 8 – Next Word Prediction Model**

Simple architecture:

Embedding → Linear → Softmax

In [ ]:
class NextWordModel(nn.Module):

    def __init__(self,vocab_size):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size+1,50)

        self.fc = nn.Linear(50,vocab_size+1)

    def forward(self,x):

        x = self.embedding(x)

        x = x.view(1,-1)

        x = self.fc(x)

        return x

**Step 9 – Local Client Training**

In [ ]:
def train_client(model,data,epochs=5):

    optimizer = optim.Adam(model.parameters(),lr=0.01)

    criterion = nn.CrossEntropyLoss()

    model.train()

    for epoch in range(epochs):

        for x,y in data:

            x = torch.tensor([x])
            y = torch.tensor([y])

            optimizer.zero_grad()

            output = model(x)

            loss = criterion(output,y)

            loss.backward()

            optimizer.step()

    return model.state_dict()

**Step 10 – Federated Averaging**

In [ ]:
def federated_average(global_model,client_weights):

    global_dict = global_model.state_dict()

    for key in global_dict.keys():

        global_dict[key] = torch.stack(
            [client_weights[i][key] for i in range(len(client_weights))]
        ).mean(0)

    global_model.load_state_dict(global_dict)

    return global_model

**Step 11 – Federated Training Loop**

In [ ]:
global_model = NextWordModel(vocab_size)

ROUNDS = 5

for r in range(ROUNDS):

    client_weights = []

    for c in range(NUM_CLIENTS):

        local_model = NextWordModel(vocab_size)

        local_model.load_state_dict(global_model.state_dict())

        weights = train_client(local_model,client_data[c])

        client_weights.append(weights)

    global_model = federated_average(global_model,client_weights)

    print("Round",r,"completed")

Round 0 completed
Round 1 completed
Round 2 completed
Round 3 completed
Round 4 completed


**Step 12 – Next Word Prediction**

In [ ]:
def predict_next(word):

    global_model.eval()

    idx = word2idx[word]

    x = torch.tensor([idx])

    output = global_model(x)

    probs = torch.softmax(output,dim=1)

    pred = torch.argmax(probs).item()

    return idx2word.get(pred,"unknown")

**Step 13 – Test Typing Suggestions**

In [ ]:
print("I →",predict_next("i"))
print("machine →",predict_next("machine"))
print("learning →",predict_next("learning"))

I → love
machine → learning
learning → protects


Mobile Device 1

     ↓
Local Text Training

     ↓
Mobile Device 2

     ↓
Local Text Training

     ↓
Mobile Device 3

     ↓
Local Text Training

     ↓
      Server

     ↓
Federated Averaging

     ↓
Global Typing Suggestion Model